# Bet sizing and risk of ruin

A trading bot with a real edge can still go broke. The edge is a claim about the average of many bets. The bankroll has to survive the ones that arrive first, in the order they arrive, and it only has to reach zero once.

This post answers two questions about that bankroll. Why does a stop-loss exist, when it cannot improve what a bot expects to make? And how should the bet move as the bankroll moves, above all while it is growing? The path runs from ruin with a flat bet, through the boundaries that end a session, to a rule that re-sizes the bet from the money in hand. Asking that rule to grow the bankroll fastest collapses it to one number, the Kelly fraction, and the last section walks back from Kelly to what a bot can actually run.

## Ruin with a flat bet

Take even-money bets: win one unit with probability $p$, lose one with probability $q = 1 - p$. Start with $u$ units, stake one unit a bet, and play forever. The textbook answer is

$$P(\text{ruin}) = \begin{cases} \left(\dfrac{q}{p}\right)^{u} & p > q \\[2mm] 1 & p \le q. \end{cases}$$

The second line gets quoted a lot, no edge means certain ruin, and it is the weaker half. It is a statement about the word *forever*: a walk with no upward drift returns to every level it has visited, so with unlimited time it visits zero, and the theorem says nothing about whether that happens on Tuesday or in ten thousand years. A bot at $p = 0.49$ with 20 units and one with 20,000 are both "ruin $= 1$", and they are not the same business.

The first line is the one that pays rent, because of what $u$ is counting. Stake $b$ units a bet instead of one and nothing about the game changes except the unit: the same bankroll is now worth $u/b$ bets,

$$P(\text{ruin}) = (q/p)^{u/b}.$$

The exponent counts losses the bankroll can absorb, not money, and the bet size is the only thing that sets it. Every extra loss that fits multiplies the ruin risk by $q/p$, which at $p = 0.52$ is $0.92$.

Stop playing forever and quit at a target $T$, and the certainty becomes an ordinary number:

$$P(\text{ruin}) = \frac{(q/p)^{u/b} - (q/p)^{T/b}}{1 - (q/p)^{T/b}} \quad (p \ne q), \qquad \frac{T - u}{T} \quad (p = q).$$

Everything below is those two formulas, first with the bet picked once and never touched, then with it allowed to move.

In [1]:
import numpy as np


def random_walks(p, u, bet, n_bets, n_paths, seed=0):
    rng = np.random.default_rng(seed)
    steps = np.where(rng.random((n_paths, n_bets)) < p, bet, -bet)
    start = np.full((n_paths, 1), float(u))
    return np.hstack([start, u + np.cumsum(steps, axis=1)])


def first_time(walks, level, below):
    crossed = walks <= level if below else walks >= level
    ever = crossed.any(axis=1)
    return ever, np.where(ever, crossed.argmax(axis=1), walks.shape[1])


def session(walks, floor, target):
    down, t_down = first_time(walks, floor, True)
    up, t_up = first_time(walks, target, False)
    stop = np.minimum(np.minimum(t_down, t_up), walks.shape[1] - 1)
    end = walks[np.arange(walks.shape[0]), stop]
    return end, stop, down & (t_down <= t_up)


def ruin_formula(p, u, bet, target=None):
    if abs(p - 0.5) < 1e-12:
        return 1.0 if target is None else (target - u) / target
    ratio = ((1 - p) / p) ** (1 / bet)
    if target is None:
        return min(1.0, ratio ** u)
    return (ratio ** u - ratio ** target) / (1 - ratio ** target)

## The cliff

Hold everything still except the bet. Bankroll 100 units, the stake picked once and never revised. Markers are simulated over 6,000 bets; the dashed line is $(q/p)^{u/b}$, ruin *eventually*, with no clock on it. Slide through the edges.

With an edge, read the left wall. The exponent is $u/b$, so halving the bet squares the ruin risk: at $p = 0.52$ a ten-unit bet ruins 45% of the time, a five-unit bet 20%, and 20% is 45% squared. The whole row runs from 73% at twenty-five units to 0.03% at one. Same edge, same money, same number of bets; only the share of the bankroll riding on each one moved, and the usable range is crowded against that wall. There is no gentle region to be slightly wrong in.

With no edge, read the dashed line: flat on 1 for every bet size. Bet size cannot change where an edgeless walk ends. What it changes is the gap between the markers and the line, and that gap is nothing but the clock: at $p = 0.50$ a one-unit bettor ruins 19% of the time inside 6,000 bets and a twenty-five-unit bettor 97% of the time, on the way to the same place. Small bets do not save an edgeless bot. They stretch the timeline, and the bankroll pays for the extra time either way.

In [2]:
import plotly.graph_objects as go

EDGES2 = [0.49, 0.50, 0.52, 0.55]
U2 = 100
BETS2 = [1, 2, 4, 5, 10, 20, 25]
N2 = 6000
PATHS2 = 600

fig = go.Figure()
for pe in EDGES2:
    sim = [float(session(random_walks(pe, U2, b, N2, PATHS2, seed=2), 0.0, np.inf)[2].mean())
           for b in BETS2]
    theory = [ruin_formula(pe, U2, b) for b in BETS2]
    show = (pe == 0.52)
    fig.add_trace(go.Scatter(x=BETS2, y=sim, mode="markers+lines", visible=show,
                             marker=dict(size=9, color="#1f77b4"),
                             line=dict(color="#1f77b4", width=1),
                             name=f"simulated, {N2} bets"))
    fig.add_trace(go.Scatter(x=BETS2, y=theory, mode="lines", visible=show,
                             line=dict(color="#888888", dash="dash"),
                             name="eventual"))

steps2 = []
for k, pe in enumerate(EDGES2):
    vis = [k == j // 2 for j in range(len(EDGES2) * 2)]
    steps2.append(dict(method="update", label=str(pe),
                       args=[{"visible": vis},
                             {"title": f"Ruin against bet size — p = {pe}, bankroll {U2}"}]))

fig.update_layout(
    sliders=[dict(active=2, currentvalue={"prefix": "win probability p = "}, steps=steps2)],
    title=f"Ruin against bet size — p = 0.52, bankroll {U2}",
    xaxis_title="bet size (units)", yaxis_title="probability of ruin",
    yaxis_range=[-0.03, 1.03], legend=dict(x=0.02, y=0.98),
    margin=dict(t=70))
fig.show()

## Why a stop-loss exists

Zero is absorbing and no other number on the line is. Two boundaries take that asymmetry away: a **stop-loss** that ends the session above zero, and a **take-win** that ends it above where you started. The walk now finishes somewhere you picked instead of somewhere it wandered.

Bankroll 100, a flat bet of 5 units, both boundaries measured in units from the start. Markers are simulated, lines are the two-barrier formula. The left panel is your chance of reaching the take-win before the stop-loss. Push the take-win out and it falls; dig the stop-loss deeper and you buy some of it back. Every point on those curves is yours to pick: bank small and often, or large and rarely.

The right panel is what none of those points move. Expected profit for a session is

$$\mathbb{E}[\text{profit}] = (p - q) \times b \times \mathbb{E}[\text{bets played}],$$

the edge, times the stake, times how long you stayed in, and boundaries touch only the last term. At $p = 0.52$ the widest setting earns the most, 55 units a session with the stop-loss at 75 and the take-win at 100, because staying in longer is how an edge gets collected. At $p = 0.50$ the panel is flat on zero. At $p = 0.48$ every setting loses and the widest loses most, 49 units, because a long session buys a losing bot nothing but more chances to lose.

So a stop-loss cannot beat the house, and with a known edge it cannot even help: it shortens the session and nothing else. Its reasons are elsewhere, and the figure holds both.

The first is that the edge is a belief, not a fact. Nobody runs a bot at a $p$ they have measured to the third decimal; they run it at a $p$ they have estimated, and the $0.48$ panel is what an estimate of $0.52$ costs over a long session when it is wrong by four points. A stop-loss caps that cost at a number chosen in advance.

The second is where a stop-loss leaves you: above zero, holding money, at a moment you chose, with the walk halted. That is the point where you can look at the bankroll you actually have and set the next bet against it. The rest of the post is about what to set it to.

In [3]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

EDGES3 = [0.48, 0.50, 0.52, 0.55]
U3 = 100
BET3 = 5
WINS3 = [10, 20, 30, 50, 75, 100]
LOSSES3 = [25, 50, 75]
COLORS3 = ["#1f77b4", "#e07b39", "#2ca02c"]
PER3 = len(LOSSES3) * 3

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    "Chance of reaching the take-win first", "Expected profit per session (units)"))

for pe in EDGES3:
    show = (pe == 0.52)
    w = random_walks(pe, U3, BET3, 1500, 1000, seed=3)
    for color, loss in zip(COLORS3, LOSSES3):
        ruin = [ruin_formula(pe, loss, BET3, loss + win) for win in WINS3]
        reach = [1 - r for r in ruin]
        ev = [r * win - (1 - r) * loss for r, win in zip(reach, WINS3)]
        sim = [1 - float(session(w, U3 - loss, U3 + win)[2].mean()) for win in WINS3]
        fig.add_trace(go.Scatter(x=WINS3, y=reach, mode="lines", visible=show,
                                 line=dict(color=color), name=f"stop-loss {loss}"),
                      row=1, col=1)
        fig.add_trace(go.Scatter(x=WINS3, y=sim, mode="markers", visible=show,
                                 marker=dict(size=7, color=color), showlegend=False),
                      row=1, col=1)
        fig.add_trace(go.Scatter(x=WINS3, y=ev, mode="lines", visible=show,
                                 line=dict(color=color), showlegend=False),
                      row=1, col=2)

fig.add_hline(y=0, line=dict(color="#bbbbbb", width=1), row=1, col=2)

steps3 = []
for k, pe in enumerate(EDGES3):
    vis = [k == j // PER3 for j in range(len(EDGES3) * PER3)]
    steps3.append(dict(method="update", label=str(pe),
                       args=[{"visible": vis},
                             {"title": f"Boundaries at p = {pe} — bankroll {U3}, flat {BET3}-unit bets"}]))

fig.update_layout(
    sliders=[dict(active=2, currentvalue={"prefix": "win probability p = "}, steps=steps3)],
    title=f"Boundaries at p = 0.52 — bankroll {U3}, flat {BET3}-unit bets",
    legend=dict(x=0.32, y=0.98), margin=dict(t=90))
fig.update_xaxes(title_text="take-win (units)", row=1, col=1)
fig.update_xaxes(title_text="take-win (units)", row=1, col=2)
fig.update_yaxes(title_text="probability", range=[0, 1.03], row=1, col=1)
fig.update_yaxes(title_text="units", row=1, col=2)
fig.show()

## Adjusting the bet as the bankroll moves

Two bots run 200 sessions at $p = 0.52$, and both open by risking a quarter of their 100 units to make a quarter.

- **Fixed**: stop-loss 25 units down, take-win 25 units up, bet 5 units. Three numbers in units, set once.
- **Proportional**: stop-loss 25% down, take-win 25% up, bet 5%. The same three numbers as fractions, re-based on the bankroll in hand at the start of every session.

Session one is the same coin flip for both; the two-barrier formula gives each a **60%** chance of reaching the take-win. What differs is what session two is played with. The fixed bot adds and subtracts 25 forever, so while it wins its bet is a shrinking share of the bankroll and while it loses a growing one, and a 25-unit stop-loss has stopped being a stop-loss by the time the bankroll is 25. Its median after 200 sessions is about **1,000**, and **20%** of its runs are at zero. The proportional bot multiplies instead of adding: median about **4,300**, and nothing at zero, because a quarter of something is never all of it. The axis is logarithmic, so read the shapes: adding bends flat, multiplying goes straight.

Drop the slider to $p = 0.51$ and the proportional bot loses money, median 26, while the fixed bot's median of 350 comes with 44% of its runs at zero. Nothing is wrong with re-basing. 5% of the bankroll is simply too large a bet for a 2% edge; the next section says why, and how large is right.

The two bots are the ends of one dial. Any rule that sets the bet from the bankroll can be written as $b = b_0 (B/B_0)^{\gamma}$: at $\gamma = 0$ it adds, at $\gamma = 1$ it multiplies. Below 1 the staked share grows as the bankroll shrinks, which is the fixed bot's death. Above 1 the share rises with every win until it passes the point where the edge no longer covers the swings, which the next figure locates. Only $\gamma = 1$ keeps the share fixed, so the question is which share.

In [4]:
import plotly.graph_objects as go

EDGES4 = [0.51, 0.52, 0.53, 0.54]
B0 = 100.0
SESSIONS = 200
PATHS4 = 3000
STEP4 = 25.0
FRAC4 = 0.25
CUSHION = 5
PER4 = 6


def run_sessions(p, seed=4):
    rng = np.random.default_rng(seed)
    p_win = 1 - ruin_formula(p, CUSHION, 1, 2 * CUSHION)
    wins = rng.random((SESSIONS, PATHS4)) < p_win
    fixed = np.full(PATHS4, B0)
    prop = np.full(PATHS4, B0)
    fx = np.empty((SESSIONS + 1, PATHS4))
    pr = np.empty((SESSIONS + 1, PATHS4))
    fx[0], pr[0] = fixed, prop
    for t in range(SESSIONS):
        fixed = np.where(fixed > 0, fixed + np.where(wins[t], STEP4, -STEP4), 0.0)
        prop = prop * np.where(wins[t], 1 + FRAC4, 1 - FRAC4)
        fx[t + 1], pr[t + 1] = fixed, prop
    return fx, pr, p_win


fig = go.Figure()
x4 = list(range(SESSIONS + 1))
summary = []
for pe in EDGES4:
    show = (pe == 0.52)
    fx, pr, p_win = run_sessions(pe)
    ruin4 = float(np.mean(fx[-1] <= 0))
    summary.append((p_win, float(np.median(fx[-1])), ruin4, float(np.median(pr[-1]))))
    for series, color, label in ((fx, "#e07b39", "fixed units"), (pr, "#1f77b4", "proportional")):
        clipped = np.clip(series, 0.1, None)
        lo = np.percentile(clipped, 10, axis=1)
        hi = np.percentile(clipped, 90, axis=1)
        mid = np.percentile(clipped, 50, axis=1)
        fig.add_trace(go.Scatter(x=x4, y=lo.tolist(), mode="lines", visible=show,
                                 line=dict(width=0), showlegend=False, hoverinfo="skip"))
        fig.add_trace(go.Scatter(x=x4, y=hi.tolist(), mode="lines", visible=show,
                                 line=dict(width=0), fill="tonexty",
                                 fillcolor="rgba(120,120,120,0.15)", showlegend=False,
                                 hoverinfo="skip"))
        fig.add_trace(go.Scatter(x=x4, y=mid.tolist(), mode="lines", visible=show,
                                 line=dict(color=color, width=2), name=label))


def title4(k):
    p_win, med_fx, ruin4, med_pr = summary[k]
    return (f"p = {EDGES4[k]}, session win rate {p_win:.0%} — "
            f"fixed: median {med_fx:,.0f}, ruin {ruin4:.0%} · "
            f"proportional: median {med_pr:,.0f}, ruin 0%")


steps4 = [dict(method="update", label=str(pe),
               args=[{"visible": [k == j // PER4 for j in range(len(EDGES4) * PER4)]},
                     {"title": title4(k)}])
          for k, pe in enumerate(EDGES4)]
fig.update_layout(
    sliders=[dict(active=1, currentvalue={"prefix": "win probability p = "}, steps=steps4)],
    title=title4(1),
    xaxis_title="sessions", yaxis_title="bankroll (units)", yaxis_type="log",
    legend=dict(x=0.02, y=0.98), margin=dict(t=90))
fig.show()

## The share that grows fastest

Stake a share $f$ of the current bankroll on every bet. A win multiplies the bankroll by $1 + f$, a loss by $1 - f$, so after many bets the bankroll is a product of those factors, and what compounds is the average of their logarithms:

$$g(f) = p \ln(1 + f) + q \ln(1 - f),$$

the growth per bet. It depends on the share and on nothing else, not on the bankroll, which is why a rule that keeps the share fixed is the only one that behaves the same at every size. Maximize it:

$$g'(f) = \frac{p}{1 + f} - \frac{q}{1 - f} = 0 \quad\Longrightarrow\quad f^{*} = p - q.$$

That is the **Kelly fraction**: stake the edge. Any other rule, flat bets included, stakes some share at every step, and its expected log bankroll after $T$ bets is at most $\ln B_0 + T\,g(f^{*})$, with equality only if the share is $f^{*}$ every time. Kelly is not one candidate among the rules that re-size from the bankroll. It is what all of them collapse to once they are asked to grow fastest.

The figure is $g$ against $f$, with three points marked. Half Kelly keeps three quarters of the growth, because for a small edge $g(k f^{*}) \approx (2k - k^{2})\,g(f^{*})$. Double Kelly sits on zero: the swings have eaten the whole edge, and past it a winning bot loses money. The proportional bot of the last section staked 5%, which is $1.25\,f^{*}$ at $p = 0.52$ and $2.5\,f^{*}$ at $p = 0.51$. That is the whole story of its median falling from 4,300 to 26.

In [5]:
import plotly.graph_objects as go

EDGES5 = [0.51, 0.52, 0.53, 0.55]
FRACTIONS = np.linspace(0.0, 0.3, 301)


def growth_rate(p, f):
    return p * np.log1p(f) + (1 - p) * np.log1p(-f)


def title5(pe):
    return f"Growth per bet against the staked share — p = {pe}, Kelly f* = {2 * pe - 1:.2f}"


def ranges5(pe):
    fstar = 2 * pe - 1
    top = growth_rate(pe, fstar) * 1e4
    return {"xaxis.range": [0, 3 * fstar], "yaxis.range": [-1.5 * top, 1.4 * top]}


fig = go.Figure()
for pe in EDGES5:
    show = (pe == 0.52)
    fstar = 2 * pe - 1
    fig.add_trace(go.Scatter(x=FRACTIONS.tolist(), y=(growth_rate(pe, FRACTIONS) * 1e4).tolist(),
                             mode="lines", visible=show, line=dict(color="#1f77b4", width=2),
                             name="growth per bet"))
    marks = [fstar / 2, fstar, 2 * fstar]
    fig.add_trace(go.Scatter(x=marks, y=[growth_rate(pe, m) * 1e4 for m in marks],
                             mode="markers+text", visible=show,
                             marker=dict(size=9, color=["#9ecae1", "#1f77b4", "#d62728"]),
                             text=["half Kelly", "Kelly", "double Kelly"],
                             textposition="top center", showlegend=False))
fig.add_hline(y=0, line=dict(color="#bbbbbb", width=1))

steps5 = [dict(method="update", label=str(pe),
               args=[{"visible": [k == j // 2 for j in range(len(EDGES5) * 2)]},
                     {"title": title5(pe), **ranges5(pe)}])
          for k, pe in enumerate(EDGES5)]
fig.update_layout(
    sliders=[dict(active=1, currentvalue={"prefix": "win probability p = "}, steps=steps5)],
    title=title5(0.52),
    xaxis_title="share of the bankroll staked per bet", yaxis_title="growth per bet (x 1e-4)",
    xaxis_range=ranges5(0.52)["xaxis.range"], yaxis_range=ranges5(0.52)["yaxis.range"],
    showlegend=False, margin=dict(t=70))
fig.show()

## The way back from Kelly

A bot does not re-size after every bet. It re-sizes at the boundaries: inside a session the bet is flat, so the staked share drifts away from $f^{*}$ as the bankroll moves, and it is only reset when a stop-loss or a take-win ends the session. The figure prices that drift, growth per bet against the width of the band between the two boundaries, for three shares written as multiples of $f^{*}$.

Every curve rises as the band narrows, because re-basing more often is resizing more often. Narrow it to a single bet and you are resizing after every bet, which is Kelly with no boundaries left at all, the dashed line. The curves climb toward it and none of them crosses it. So take-win and stop-loss are not a strategy layered on top of bet sizing. They are the schedule on which you are allowed to do it, and the schedule costs growth in proportion to how rarely it runs.

Kelly is the best share at every width, and it is still not the share to run. $f^{*} = p - q$ doubles every error in $p$: a bot that believes $0.52$ when the truth is $0.51$ is at double Kelly, on zero growth, without a single bad decision. The reason that justified the stop-loss, that the edge is a belief, justifies staking under it. Half Kelly at a believed 52% is exactly Kelly at a true 51%: it gives up a quarter of the growth to survive being wrong by a full point.

In [6]:
import plotly.graph_objects as go

MULTIPLES = [(0.5, "half Kelly", "#9ecae1"), (1.0, "Kelly", "#1f77b4"),
             (2.0, "double Kelly", "#d62728")]
PER6 = len(MULTIPLES) + 1


def band_growth(p, f, band):
    ratio = (1 - p) / p
    n = band / f
    p_win = 1 / (1 + ratio ** n)
    expected_bets = n * (2 * p_win - 1) / (2 * p - 1)
    g = p_win * np.log(1 + band) + (1 - p_win) * np.log(1 - band)
    return g / expected_bets


fig = go.Figure()
for pe in EDGES4:
    show = (pe == 0.52)
    fstar = 2 * pe - 1
    for mult, label, color in MULTIPLES:
        f = mult * fstar
        bands = np.linspace(2 * f, 0.9, 90)
        fig.add_trace(go.Scatter(x=bands.tolist(), y=(band_growth(pe, f, bands) * 1e4).tolist(),
                                 mode="lines", line=dict(color=color), visible=show, name=label))
    kelly_g = growth_rate(pe, fstar) * 1e4
    fig.add_trace(go.Scatter(x=[0, 0.9], y=[kelly_g, kelly_g], mode="lines", visible=show,
                             line=dict(color="#888888", dash="dash"), name="continuous Kelly"))

steps6 = [dict(method="update", label=str(pe),
               args=[{"visible": [k == j // PER6 for j in range(len(EDGES4) * PER6)]},
                     {"title": f"Growth per bet against band width — p = {pe}"}])
          for k, pe in enumerate(EDGES4)]
fig.update_layout(
    sliders=[dict(active=1, currentvalue={"prefix": "win probability p = "}, steps=steps6)],
    title="Growth per bet against band width — p = 0.52",
    xaxis_title="band width (fraction of bankroll)", yaxis_title="growth per bet (x 1e-4)",
    legend=dict(x=0.70, y=0.98), margin=dict(t=70))
fig.show()

## Conclusion

Two questions, two answers.

A stop-loss exists because the edge is a belief and because zero is absorbing. It cannot raise what a bot expects to make; expected profit is edge times stake times bets played, and a boundary only shortens the last term. What it does is cap what a wrong belief about the edge costs per session, and end the walk above zero at a moment of your choosing, which is the moment the bet can be set again.

The bet should be set as a share of the bankroll in hand, and the share that grows fastest is the edge itself, $f^{*} = p - q$. Every rule that re-sizes from the bankroll collapses to that one once it is asked to grow fastest, and every rule that re-sizes less often than every bet, a session between a stop-loss and a take-win included, is Kelly on a schedule, paying for the gaps in growth. Run it below Kelly, because the error in your $p$ is doubled in your $f$, and let the boundaries be the moments you re-base.

**[In-browser notebook →](/lite/notebooks/index.html?path=2026-09-24-bet-sizing-and-risk-of-ruin.ipynb)**